In [1]:
# ============================================================
# Performance Benchmark
# ============================================================

import os
import time
import joblib
import psutil
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# Measure Model Loading Time
# ============================================================

start_time = time.perf_counter()

preprocessor = joblib.load("../models/preprocessor.pkl")
best_xgb = joblib.load("../models/xgb.pkl")
best_lgb = joblib.load("../models/lightgbm.pkl")
best_mlp = joblib.load("../models/mlp.pkl")
calibrated_model = joblib.load("../models/calibrated.pkl")
best_threshold = joblib.load("../models/threshold.pkl")

end_time = time.perf_counter()

loading_time = end_time - start_time

print(f"Model Loading Time : {loading_time:.4f} seconds")

Model Loading Time : 1.7492 seconds


In [3]:
# ============================================================
# Load Sample Data
# ============================================================

sample = pd.read_csv("../samples/sample_input.csv")

print(f"Loaded {len(sample)} sample records.")

Loaded 10 sample records.


In [4]:
# ============================================================
# Measure Single Prediction Speed
# ============================================================

single_sample = sample.iloc[[0]]

start = time.perf_counter()

sample_processed = preprocessor.transform(single_sample)

xgb_prob = best_xgb.predict_proba(sample_processed)[:, 1]
lgb_prob = best_lgb.predict_proba(sample_processed)[:, 1]
mlp_prob = best_mlp.predict_proba(sample_processed)[:, 1]

meta_features = pd.DataFrame({

    "XGB": xgb_prob,
    "LGBM": lgb_prob,
    "MLP": mlp_prob

})

probability = calibrated_model.predict_proba(meta_features)[:, 1][0]

prediction = int(probability >= best_threshold)

end = time.perf_counter()

prediction_time = (end - start) * 1000

print(f"Prediction Time : {prediction_time:.2f} ms")
print(f"Probability     : {probability:.6f}")
print(f"Prediction      : {prediction}")

Prediction Time : 17.77 ms
Probability     : 0.339466
Prediction      : 0


d:\Projects\Diabetes-Risk-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
# ============================================================
# Measure Batch Prediction Speed
# ============================================================

batch = pd.concat([sample] * 100, ignore_index=True)

start = time.perf_counter()

batch_processed = preprocessor.transform(batch)

xgb_prob = best_xgb.predict_proba(batch_processed)[:, 1]
lgb_prob = best_lgb.predict_proba(batch_processed)[:, 1]
mlp_prob = best_mlp.predict_proba(batch_processed)[:, 1]

meta_features = pd.DataFrame({

    "XGB": xgb_prob,
    "LGBM": lgb_prob,
    "MLP": mlp_prob

})

_ = calibrated_model.predict_proba(meta_features)

end = time.perf_counter()

batch_time = (end - start) * 1000

print(f"Processed {len(batch)} records")
print(f"Batch Prediction Time : {batch_time:.2f} ms")
print(f"Average per Record    : {batch_time / len(batch):.4f} ms")

Processed 1000 records
Batch Prediction Time : 19.91 ms
Average per Record    : 0.0199 ms


d:\Projects\Diabetes-Risk-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
# ============================================================
# Measure Memory Usage
# ============================================================

process = psutil.Process(os.getpid())

memory = process.memory_info().rss / (1024 ** 2)

print(f"Current Memory Usage : {memory:.2f} MB")

Current Memory Usage : 319.48 MB


In [8]:
# ============================================================
# Model File Sizes
# ============================================================

model_files = [

    "preprocessor.pkl",
    "xgb.pkl",
    "lightgbm.pkl",
    "mlp.pkl",
    "meta.pkl",
    "calibrated.pkl",
    "threshold.pkl",
    "feature_names.pkl",
    "target_labels.pkl",
    "metadata.json"

]

print("=" * 60)

total_size = 0

for file in model_files:

    path = os.path.join("../models", file)

    size = os.path.getsize(path) / (1024 * 1024)

    total_size += size

    print(f"{file:<22} {size:.2f} MB")

print("=" * 60)
print(f"Total Model Size      {total_size:.2f} MB")

preprocessor.pkl       0.01 MB
xgb.pkl                0.69 MB
lightgbm.pkl           1.86 MB
mlp.pkl                0.32 MB
meta.pkl               0.00 MB
calibrated.pkl         0.02 MB
threshold.pkl          0.00 MB
feature_names.pkl      0.00 MB
target_labels.pkl      0.00 MB
metadata.json          0.00 MB
Total Model Size      2.90 MB


In [9]:
# ============================================================
# Performance Summary
# ============================================================

print("=" * 60)
print("Production Performance Summary")
print("=" * 60)

print(f"Model Loading Time     : {loading_time:.4f} sec")
print(f"Single Prediction      : {prediction_time:.2f} ms")
print(f"Batch Prediction (1000): {batch_time:.2f} ms")
print(f"Memory Usage           : {memory:.2f} MB")
print(f"Total Model Size       : {total_size:.2f} MB")

print("=" * 60)

Production Performance Summary
Model Loading Time     : 1.7492 sec
Single Prediction      : 17.77 ms
Batch Prediction (1000): 19.91 ms
Memory Usage           : 319.48 MB
Total Model Size       : 2.90 MB
